# Hexagon historical age from Wikidata singular elements

Determines the **historical age of each H3 hexagon** from singular Wikidata elements (churches, convents, palaces, gates, fountains, bridges…) and their construction dates. Each element is assigned to its hexagon; the cell receives the year of its oldest element and a weighted **historical-anchor score** (older elements weigh more).

Input: `query.geojson` (Wikidata POIs, in the MQUEA folder). Output: `wikidata_historical_age_h3.csv` + map `imagenes/wikidata_historical_age_EN.png`.

In [ ]:
import json, re
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
import h3

QUERY = 'query.geojson'                       # Wikidata singular elements (MQUEA folder)
GPKG  = 'madrid_morfologia_h3_v10.gpkg'       # H3 hexagon grid
RES   = 9

raw = json.load(open(QUERY, encoding='utf-8'))
rows = []
for ft in raw['features']:
    g = ft.get('geometry') or {}
    if g.get('type') != 'Point':
        continue
    lon, lat = g['coordinates'][:2]
    p = ft['properties']
    rows.append({'item': p.get('item',''), 'label': p.get('itemLabel',''), 'type': p.get('typeLabel',''),
                 'inception': p.get('inception'), 'start': p.get('start'), 'opening': p.get('opening'),
                 'lon': lon, 'lat': lat})
df = pd.DataFrame(rows)
print('point features:', len(df), '| unique items:', df['item'].nunique())

In [ ]:
# Preferred date: inception > start > opening ; then deduplicate by item (keep the dated row)
df['date_raw'] = df['inception']
for alt in ['start', 'opening']:
    miss = df['date_raw'].isna() | (df['date_raw'] == '')
    df.loc[miss, 'date_raw'] = df.loc[miss, alt]
df['_has_date'] = df['date_raw'].notna() & (df['date_raw'] != '')
df = df.sort_values(['item','_has_date'], ascending=[True, False]).drop_duplicates('item', keep='first').copy()

def parse_year(s):
    if pd.isna(s) or s == '':
        return np.nan
    m = re.search(r'([+-]?\d{3,4})', str(s))      # handles 1623-.., +1623-.., 0855-..
    if not m:
        return np.nan
    y = int(m.group(1))
    return y if 1 <= y <= 2025 else np.nan

df['year'] = df['date_raw'].apply(parse_year)
dfy = df[df['year'].notna()].copy()
print('with valid year:', len(dfy), '| range', int(dfy.year.min()), '-', int(dfy.year.max()))

In [ ]:
# Historical periods + assign each element to its H3 res-9 hexagon
def period(y):
    if y < 1500:  return 'pre1500'
    if y < 1700:  return '1500_1700'
    if y < 1860:  return '1700_1860'
    if y < 1940:  return '1860_1940'
    return 'post1940'
dfy['wiki_period'] = dfy['year'].apply(period)
dfy['h3_id'] = [h3.latlng_to_cell(la, lo, RES) for la, lo in zip(dfy['lat'], dfy['lon'])]

# Aggregate per hexagon (the mean is NOT the main variable; oldest element + weighted score are)
agg = dfy.groupby('h3_id')['year'].agg(
    wiki_n_total='count', wiki_oldest_year='min', wiki_year_mean='mean',
    wiki_year_p10=lambda x: np.percentile(x, 10)).reset_index()
pc = dfy.pivot_table(index='h3_id', columns='wiki_period', values='year', aggfunc='count', fill_value=0).reset_index()
pc.columns = ['h3_id' if c == 'h3_id' else f'wiki_count_{c}' for c in pc.columns]
hex_wiki = agg.merge(pc, on='h3_id', how='left')
for c in ['wiki_count_pre1500','wiki_count_1500_1700','wiki_count_1700_1860','wiki_count_1860_1940','wiki_count_post1940']:
    if c not in hex_wiki: hex_wiki[c] = 0

# Weighted historical-anchor score: the older the element, the heavier its weight
hex_wiki['historical_anchor_score'] = (4*hex_wiki['wiki_count_pre1500'] + 3*hex_wiki['wiki_count_1500_1700']
    + 2*hex_wiki['wiki_count_1700_1860'] + 1*hex_wiki['wiki_count_1860_1940'])
hex_wiki['historical_anchor_score_log'] = np.log1p(hex_wiki['historical_anchor_score'])
for dummy, cols in [('wiki_pre1500_dummy', ['wiki_count_pre1500']),
                    ('wiki_pre1700_dummy', ['wiki_count_pre1500','wiki_count_1500_1700']),
                    ('wiki_pre1860_dummy', ['wiki_count_pre1500','wiki_count_1500_1700','wiki_count_1700_1860'])]:
    hex_wiki[dummy] = (hex_wiki[cols].sum(axis=1) > 0).astype(int)
print('hexagons with anchors:', len(hex_wiki))
print('  pre-1500:', int(hex_wiki.wiki_pre1500_dummy.sum()),
      '| pre-1700:', int(hex_wiki.wiki_pre1700_dummy.sum()),
      '| pre-1860:', int(hex_wiki.wiki_pre1860_dummy.sum()))

In [ ]:
# Merge onto the hexagon grid (match by H3 id of each hexagon centroid -> robust) and save
gdf = gpd.read_file(GPKG).to_crs(25830)
cent = gdf.geometry.centroid.to_crs(4326)
gdf['_h3'] = [h3.latlng_to_cell(c.y, c.x, RES) for c in cent]
gdf = gdf.drop(columns=[c for c in gdf.columns if c.startswith('wiki_') or c.startswith('historical_anchor')], errors='ignore')
gdf = gdf.merge(hex_wiki, left_on='_h3', right_on='h3_id', how='left')
zc = [c for c in gdf.columns if c.startswith('wiki_count_') or c.endswith('_dummy') or c.startswith('historical_anchor')]
gdf[zc] = gdf[zc].fillna(0)
print('grid hexagons with Wikidata data:', int(gdf['wiki_n_total'].notna().sum()), 'of', len(gdf))

keep = ['hex_id','wiki_n_total','wiki_oldest_year','wiki_year_mean','wiki_year_p10',
        'wiki_count_pre1500','wiki_count_1500_1700','wiki_count_1700_1860','wiki_count_1860_1940','wiki_count_post1940',
        'wiki_pre1500_dummy','wiki_pre1700_dummy','wiki_pre1860_dummy','historical_anchor_score','historical_anchor_score_log']
keep = [c for c in keep if c in gdf.columns]
gdf[keep].to_csv('wikidata_historical_age_h3.csv', index=False)
print('saved -> wikidata_historical_age_h3.csv')

In [ ]:
# Maps (English) + effect size vs morphology label
disp = gdf[gdf['cat_n_buildings'].fillna(0) >= 5].copy()
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
disp.plot(ax=axes[0], color='#EEEEEE', linewidth=0)
disp[disp['historical_anchor_score'] > 0].plot(column='historical_anchor_score_log', cmap='magma_r',
    legend=True, legend_kwds={'label':'Historical-anchor score (log)','shrink':0.6}, ax=axes[0], linewidth=0)
axes[0].set_title('Wikidata historical-anchor score per hexagon\n(weighted count of singular elements; older = heavier)', fontsize=12)
disp.plot(column='wiki_oldest_year', cmap='RdYlBu', legend=True,
    legend_kwds={'label':'Oldest element year','shrink':0.6}, missing_kwds={'color':'#EEEEEE'}, ax=axes[1], linewidth=0)
axes[1].set_title('Wikidata — oldest singular element per hexagon', fontsize=12)
for ax in axes: ax.set_axis_off(); ax.set_aspect('equal')
plt.tight_layout(); plt.savefig('imagenes/wikidata_historical_age_EN.png', dpi=170, bbox_inches='tight'); plt.show()

if 'barrio_label' in gdf.columns:
    lab = gdf[gdf['barrio_label'].isin([0, 1])]
    for col in ['wiki_oldest_year', 'historical_anchor_score']:
        e = lab[lab.barrio_label == 0][col].dropna(); p = lab[lab.barrio_label == 1][col].dropna()
        if len(e) > 2 and len(p) > 2:
            d = (e.mean() - p.mean()) / np.sqrt((e.std()**2 + p.std()**2) / 2)
            print(f'Cohen d {col}: {d:+.3f}  (esp n={len(e)}, plan n={len(p)})')